In [ ]:
import os, sys
print("cwd:", os.getcwd())
print("sys.path[:5]:", sys.path[:5])

In [ ]:
%cd D:\shahnawaz\uva\main

In [ ]:
%load_ext autoreload
%autoreload 2

from src.data_loader import DataLoader
from src.data_analyzer import DataAnalyzer
from src.constant_manager import ConstantManager
from src.data_cleaner import DataCleaner
from src.feature_renamer import FeatureRenamer
from src.scaler import Scaler

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# file names declared globally for easy access
RAW_DATA_FILE = 'rcci_data_v4_5.xlsx'
PD_DATA_FILE = 'rcci_cleaned_data_v4_5.parquet'

In [ ]:
pd_loader = DataLoader(file_path=PD_DATA_FILE)
pd_df = pd_loader.load_data()

pd_df.head()

In [ ]:
scaler_x = Scaler()
scaler_y = Scaler()

if_names = ConstantManager().RAW_REDUCED_INPUT_COLUMNS
of_names = ConstantManager().RAW_OUTPUT_COLUMNS

scaled_df = pd_df[if_names + of_names].copy()
scaled_df[if_names] = scaler_x.fit_transform(scaled_df, if_names)
scaled_df[of_names] = scaler_y.fit_transform(scaled_df, of_names)


In [ ]:
scaled_df.head()

In [ ]:
case1 = scaled_df.iloc[0]
print("Case 1 (scaled):")
print(case1)

In [ ]:
case1_inputs_original = scaler_x.inverse_transform(case1[if_names], dim_idx=range(len(if_names)))
print("\nCase 1 Inputs (original scale): {}".format(case1_inputs_original))
    
case1_outputs_original = scaler_y.inverse_transform(case1[of_names], dim_idx=range(len(of_names)))
print("\nCase 1 Outputs (original scale): {}".format(case1_outputs_original))

In [ ]:
data_analyzer = DataAnalyzer(scaled_df)
estimated_noise_levels_scaled, estimated_residuals_scaled = data_analyzer.estimate_noise_levels(if_names, of_names)
print("Estimated noise levels in output features after scaling:", estimated_noise_levels_scaled)
print("Estimated residuals in output features after scaling:", estimated_residuals_scaled)

In [ ]:
import torch
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from torch.quasirandom import SobolEngine

from src.soed.dynamic_gp import DynamicGP

from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.exceptions import ModelFittingError
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
import gpytorch

warnings.filterwarnings("ignore")
plt.style.use("bmh")
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

# ==========================================
# 2. Multi-Step Path Planner (Unchanged)
# ==========================================
class MultiStep_sOED_Agent:
    def __init__(self, bounds):
        self.bounds = bounds
        self.d = bounds.shape[1] 
        self.model, self.likelihood = None, None
        self.X, self.Y = None, None

    def fit_data(self, X, Y):
        if isinstance(X, (pd.DataFrame, pd.Series)): X = X.values
        if isinstance(Y, (pd.DataFrame, pd.Series)): Y = Y.values
        
        self.X = torch.as_tensor(X, dtype=torch.float64)
        self.Y = torch.as_tensor(Y, dtype=torch.float64).view(-1, 1)

        # self.likelihood = gpytorch.likelihoods.GaussianLikelihood(
        #     noise_constraint=gpytorch.constraints.Interval(1e-4, 1)
        # )

        self.likelihood = gpytorch.likelihoods.GaussianLikelihood(
            noise_constraint=gpytorch.constraints.Interval(1e-3, 1.0)
        )
        
        # self.likelihood.noise = torch.tensor(0.00154353)
        # self.likelihood.raw_noise.requires_grad = False  

        self.model = DynamicGP(self.X, self.Y, self.likelihood, self.bounds)
        
        self.model.train()
        self.likelihood.train()
        mll = ExactMarginalLogLikelihood(self.likelihood, self.model)
        
        with gpytorch.settings.cholesky_jitter(1e-4):
            try:
                fit_gpytorch_mll(mll)
            except ModelFittingError:
                print("Warning: L-BFGS-B optimizer failed. Falling back to Adam...")
                optimizer = torch.optim.Adam(self.model.parameters(), lr=0.05)
                for _ in range(150):
                    optimizer.zero_grad()
                    output = self.model(*self.model.train_inputs)
                    loss = -mll(output, self.model.train_targets)
                    loss.backward()
                    optimizer.step()
        
        self.model.eval()
        self.likelihood.eval()

        # --- ADD THESE LINES TO PRINT THE LEARNED PARAMETERS ---
        with torch.no_grad():
            # Get the output scale (variance of the data overall)
            output_scale = self.model.covar_module.outputscale.item()
            
            # Get the lengthscales (base_kernel is the RBFKernel inside the ScaleKernel)
            lengthscales = self.model.covar_module.base_kernel.lengthscale.squeeze().tolist()
            
            # Get the learned noise
            noise = self.likelihood.noise.item()

            print("\n--- Model Fitting Complete ---")
            print(f"Learned Output Scale: {output_scale:.4f}")
            print(f"Learned Noise: {noise:.6f}")
            if isinstance(lengthscales, list):
                for i, ls in enumerate(lengthscales):
                    print(f"Learned Lengthscale for Feature {i} ({self.bounds.shape[1]} total): {ls:.4f}")
            else:
                print(f"Learned Lengthscale: {lengthscales:.4f}")
            print("------------------------------\n")

    def plan_multistep_batch(self, current_location, q_steps=3, num_scenarios=200, w_dist=1.0):
        sobol = SobolEngine(dimension=self.d * q_steps, scramble=True)
        raw_samples = sobol.draw(num_scenarios)
        
        paths_01 = raw_samples.view(num_scenarios, q_steps, self.d)
        range_x = self.bounds[1] - self.bounds[0]
        paths = self.bounds[0] + (range_x * paths_01)

        noise_var = self.likelihood.noise.item()

        with torch.no_grad():
            post = self.model.posterior(paths)
            covars = post.distribution.covariance_matrix
            
            I = torch.eye(q_steps, dtype=covars.dtype, device=covars.device)
            matrix_to_det = I + (covars / noise_var)
            
            try:
                Ls = torch.linalg.cholesky(matrix_to_det)
                igs = Ls.diagonal(dim1=-2, dim2=-1).log().sum(dim=-1)
            except RuntimeError:
                igs = 0.5 * torch.linalg.slogdet(matrix_to_det)[1]

            curr_loc = current_location.squeeze()
            d_start = torch.norm(paths[:, 0, :] - curr_loc, dim=-1)
            
            if q_steps > 1:
                d_steps = torch.norm(paths[:, 1:, :] - paths[:, :-1, :], dim=-1).sum(dim=-1)
            else:
                d_steps = torch.zeros_like(d_start)
                
            total_dist = d_start + d_steps
            scores = igs - (w_dist * total_dist)

            max_idx = torch.argmax(scores)
            best_path = paths[max_idx]

        return best_path

# ==========================================
# 3. Static Recommendation Dashboard
# ==========================================
def plot_recommendations(agent, optimal_path, bounds, feature_names):
    """Generates a static 1x2 plot of the GP Mean, Variance, and suggested next steps."""
    res = 50 
    d = bounds.shape[1]
    
    # Grid for visualization (Assumes first 2 dimensions for plotting)
    X1, X2 = torch.meshgrid(
        torch.linspace(bounds[0, 0].item(), bounds[1, 0].item(), res),
        torch.linspace(bounds[0, 1].item(), bounds[1, 1].item(), res), indexing="xy"
    )

    x_grid = torch.zeros(res * res, d)
    x_grid[:, 0] = X1.flatten()
    x_grid[:, 1] = X2.flatten()
    # If d > 2, keep the other dimensions at their baseline (0 or mean) for the surface plot
    
    with torch.no_grad():
        post = agent.model.posterior(x_grid)
        mean = post.mean.squeeze(-1).numpy().reshape(res, res)
        var = post.variance.squeeze(-1).numpy().reshape(res, res)

    fig = plt.figure(figsize=(24, 12))
    gs = gridspec.GridSpec(1, 2, wspace=0.1) 
    ax_mean = fig.add_subplot(gs[0, 0], projection='3d')
    ax_var = fig.add_subplot(gs[0, 1], projection='3d')

    f0, f1 = feature_names[0], feature_names[1]
    curr_loc = agent.X[-1].numpy()
    path = optimal_path.numpy()
    
    # Connect current location to the planned path
    full_path_X = np.vstack([curr_loc, path])
    path_Z_var = [var.max()] * len(full_path_X)

    # --- LEFT (Mean) ---
    ax_mean.plot_surface(X1.numpy(), X2.numpy(), mean, cmap='viridis', alpha=0.4, edgecolor='none')
    ax_mean.scatter(agent.X[:, 0].numpy(), agent.X[:, 1].numpy(), agent.Y.flatten().numpy(), 
                    c='k', s=40, label="Historical Data")
    ax_mean.set_title(f"Predictive Mean Surface\n{f0} vs {f1}")

    # --- RIGHT (Variance) ---
    ax_var.plot_surface(X1.numpy(), X2.numpy(), var, cmap='plasma', alpha=0.6, edgecolor='none')
    ax_var.plot(full_path_X[:, 0], full_path_X[:, 1], path_Z_var, c='darkgreen', linestyle='--', linewidth=2, label="Suggested Path")
    ax_var.scatter(path[:, 0], path[:, 1], [var.max()]*len(path), c='orange', s=100, edgecolors='k', label="Suggested Experiments")
    ax_var.scatter(path[0, 0], path[0, 1], var.max(), c='r', marker='*', s=300, edgecolors='none', label="Do This Next")
    ax_var.set_title("Predictive Variance & Suggested Next Steps")

    for ax in [ax_mean, ax_var]:
        # ax.view_init(elev=55, azim=-70)
        ax.view_init(elev=35, azim=-45)
        # ax.legend(loc="upper left")
        ax.set_xlabel(f0); ax.set_ylabel(f1)
        ax.set_facecolor('white')
        ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
        # ax.grid(False)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. Main Execution
# ==========================================
if __name__ == "__main__":
    # Ensure your bounds match your exact input feature dimensions
    # E.g., if you have 2 features, bounds should be shape [2, 2]
    my_bounds = torch.tensor([[-2.0, -2.0], [3.0, 3.0]]) 
    input_features = ['Boost pressure', 'Mass1']
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df['IEMP']
    # -----------------------------------------------------

    print("1. Initializing Agent and fitting historical data...")
    agent = MultiStep_sOED_Agent(my_bounds)
    agent.fit_data(inputs, output)
    
    print("2. Calculating the most informative next experiments...")
    # Assume the last row of your inputs is the "current state" of the system
    current_loc = agent.X[-1:] 
    
    # Get the next 3 recommended steps
    q_horizon = 3
    optimal_path = agent.plan_multistep_batch(current_location=current_loc, q_steps=q_horizon, w_dist=1.5)
    
    print("\n--- RECOMMENDED NEXT EXPERIMENTS ---")
    for step_idx, setting in enumerate(optimal_path):
        print(f"Step {step_idx + 1}: {setting.numpy()}")
        
    print("\n3. Rendering visualization...")
    plot_recommendations(agent, optimal_path, my_bounds, input_features)

In [ ]:
import torch
import pandas as pd
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.animate_plot import MultiOutputSliceAnimator

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":

    my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0]]) 
    input_features = ['Boost pressure', 'Mass1', 'Mass2', 'IVO']
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df[['IEMP', 'Nox']]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)

    
    
    anim = MultiOutputSliceAnimator(
        agent,
        path,
        dim_x=1,
        dim_y=2,
        fixed_at="current"
    )
    plt.ion()
    anim.animate(save_path="soed_path.gif", fps=2)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
import plotly.express as px

df_path = pd.DataFrame(
    inverse_transformed_path,
    columns=input_features
)

fig = px.parallel_coordinates(
    df_path,
    dimensions=input_features,
    color=df_path.index
)

fig.show()


In [ ]:
import torch
import pandas as pd
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.animate_plot import MultiOutputSliceAnimator

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":

    my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]]) 
    input_features = ConstantManager().RAW_REDUCED_INPUT_COLUMNS
    output_features = ConstantManager().RAW_OUTPUT_COLUMNS
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df[output_features]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
scaler_x_4 = Scaler()
scaler_y_4 = Scaler()

if_names_4 = ['Boost pressure', 'Mass1', 'Mass2', 'IVO']

scaled_df_4 = pd_df[if_names_4].copy()
scaled_df_4[if_names_4] = scaler_x_4.fit_transform(scaled_df_4, if_names_4)

In [ ]:
unscaled_bounds = {
    'Engine_speed': [990, 1010], 
    'Boost pressure': [1, 4],
    'Mass1': [2.5, 35],
    'Mass2': [0, 5],
    'SOI1': [290, 290],
    'SOI2': [30, 100],
    'IVO': [340, 460],
    'IVC': [480, 600],
    'EVO': [120, 220],
    'EVC': [260, 370]
}

In [ ]:

scaled_bounds = {}
for feature, (low, high) in unscaled_bounds.items():
    if feature in if_names_4:
        low_scaled = scaler_x_4.inverse_transform(low, feature)
        high_scaled = scaler_x_4.inverse_transform(high, feature)
        scaled_bounds[feature] = [low_scaled, high_scaled]
    else:
        scaled_bounds[feature] = [0.0, 0.0]  # or some default value for non-input features

In [ ]:
def find_bounds(df, feature_names, margin=0.5):
    bounds = []
    for feature in feature_names:
        col_min = df[feature].min()
        col_max = df[feature].max()
        col_range = col_max - col_min
        bounds.append([col_min - margin * col_range, col_max + margin * col_range])
    return torch.tensor(bounds, dtype=torch.float64)

bounds = find_bounds(scaled_df_4, if_names_4)
print("Calculated bounds for input features:")
for idx, feature in enumerate(if_names_4):
    print(f"{feature}: [{bounds[idx][0]:.4f}, {bounds[idx][1]:.4f}]")

In [ ]:
print("Bounds shape:", bounds.shape)
print(bounds)

# convert bounds [lower, upper] to [[lower1, lower2, ...], [upper1, upper2, ...]]
new_bounds = torch.stack([bounds[:, 0], bounds[:, 1]], dim=0)
print("Reformatted bounds shape:", new_bounds.shape)
print(new_bounds)

In [ ]:
import torch
import pandas as pd
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.animate_plot import MultiOutputSliceAnimator

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":


    found_bounds = find_bounds(scaled_df_4, if_names_4)
    my_bounds = torch.stack([found_bounds[:, 0], found_bounds[:, 1]], dim=0)
    # my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0]]) 
    input_features = if_names_4
    output_features = ConstantManager().RAW_OUTPUT_COLUMNS


    mass_idx_1 = input_features.index("Mass1")
    mass_idx_2 = input_features.index("Mass2")
    
    for idx in [mass_idx_1, mass_idx_2]:
        # Calculate what 0.0 kg maps to in the scaled space
        # Formula: Z = (X - mean) / scale
        mean_val = scaler_x_4.scaler.mean_[idx]
        scale_val = scaler_x_4.scaler.scale_[idx]
        scaled_zero = (0.0 - mean_val) / scale_val
        
        # Update the lower bound (my_bounds[0]) to be AT LEAST scaled_zero
        # This prevents SobolEngine from ever proposing a point below 0 kg
        my_bounds[0, idx] = torch.max(my_bounds[0, idx], torch.tensor(scaled_zero, dtype=torch.float64))

    print(f"Bounds after ensuring non-negativity for Mass1 and Mass2: {my_bounds[0]} to {my_bounds[1]}")

    # print the bounds in original scale for better interpretability
    print("\nBounds in original scale:")
    for idx, feature in enumerate(input_features):
        original_lower = scaler_x_4.inverse_transform(my_bounds[0, idx].item(), idx)
        original_upper = scaler_x_4.inverse_transform(my_bounds[1, idx].item(), idx)
        print(f"{feature}: [{original_lower:.4f}, {original_upper:.4f}]")
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df_4[input_features]
    output = scaled_df[output_features]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    # print details about the fitted model
    print("\n--- Model Fitting Complete ---")
    for idx, model in enumerate(agent.models):
        print("\n --------------------------------------------------")
        print(f"OutputModel {idx}: {output_features[idx]}")
        print(f"Learned Output Scale: {model.covar_module.outputscale.item():.4f}")
        print(f"Learned Noise: {model.likelihood.noise.item():.6f}")
        lengthscales = model.covar_module.base_kernel.lengthscale.squeeze().tolist()
        print(f"Learned Lengthscales: {lengthscales}")
        print(" --------------------------------------------------")

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
inverse_transformed_path = []
for p in path:
    original_values = []
    for dim_idx, dim_name in enumerate(input_features):
        original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
        original_values.append(original_value)
        print(f"{dim_name}: {original_value:.10f}", end=" | ")
    inverse_transformed_path.append(original_values)
    print()